# Merge flag_3 (13 Oktober) ke cleaning.db
Alur: backup DB → tambah kolom `flag_3_new` (kosong) → baca CSV → update/insert per baris → `flag_3_new = 1` untuk semua baris CSV.

In [22]:
import sqlite3, shutil
from pathlib import Path
import pandas as pd

BASE_DIR = Path.cwd()
DB_PATH = (BASE_DIR / "cleaning.db").resolve()
CSV_PATH = BASE_DIR / "cleaning_usaha_flag_3_13_oktober.csv"  # sesuaikan lokasi CSV

TABEL = "usaha"
KEYS = ["assignment_id", "nama_usaha"]
STATUS_COL = "status_penyelesaian"
KET_COL = "keterangan"
UPDATED_COL = "update_at"

assert DB_PATH.exists(), f"Database tidak ditemukan: {DB_PATH}"
assert CSV_PATH.exists(), f"CSV tidak ditemukan: {CSV_PATH}"
print("DB :", DB_PATH)
print("CSV:", CSV_PATH)


DB : /home/boms/Documents/2026/Sensus Ekonomi/cleaning_data/3 Oktober 2026/Mikro/RantabDBNAS/cleaning.db
CSV: /home/boms/Documents/2026/Sensus Ekonomi/cleaning_data/3 Oktober 2026/Mikro/RantabDBNAS/cleaning_usaha_flag_3_13_oktober.csv


In [31]:
# khusus Untuk melakukan back up 

try:
    conn.close()
except Exception:
    pass

shutil.copy2(backup, DB_PATH)
print("DB dikembalikan dari:", backup)

DB dikembalikan dari: /home/boms/Documents/2026/Sensus Ekonomi/cleaning_data/3 Oktober 2026/Mikro/RantabDBNAS/cleaning_backup_sebelum_flag3.db


In [35]:
# Backup dulu (sekali saja), lalu buka DB
tanggal = date
backup = DB_PATH.with_name(DB_PATH.stem + "_backup_{tanggal}.db")
if not backup.exists():
    shutil.copy2(DB_PATH, backup)
    print("Backup dibuat:", backup)
else:
    print("Backup sudah ada:", backup)

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

info = cur.execute(f'PRAGMA table_info("{TABEL}")').fetchall()
db_cols = [r[1] for r in info]
pk = [r for r in info if r[5] > 0]
pk_cols = [r[1] for r in pk] if len(pk) == 1 and (pk[0][2] or "").upper() == "INTEGER" else []


Backup sudah ada: /home/boms/Documents/2026/Sensus Ekonomi/cleaning_data/3 Oktober 2026/Mikro/RantabDBNAS/cleaning_backup_sebelum_flag3.db
Total baris DB: 1850931
PK autoincrement: []
updated_at tersedia di DB: True
level_2_code tersedia di DB: True


In [36]:
df = pd.read_csv(CSV_PATH, dtype=str)
# Rapikan spasi pada nama kolom CSV agar pencocokan dengan kolom DB konsisten
df.columns = df.columns.str.strip()

# level_2_code = 4 digit pertama dari level_4_full_code.
# Jika level_4_full_code kosong/tidak tersedia, gunakan level_6_full_code sebagai cadangan.
def bersihkan_kode(series):
    return (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

l4 = bersihkan_kode(df["level_4_full_code"]) if "level_4_full_code" in df.columns else pd.Series("", index=df.index, dtype="string")
l6 = bersihkan_kode(df["level_6_full_code"]) if "level_6_full_code" in df.columns else pd.Series("", index=df.index, dtype="string")
kode_sumber = l4.where(l4 != "", l6)
df["level_2_code"] = kode_sumber.str[:4].where(kode_sumber.str.len() >= 4, None)

print("Ukuran CSV:", df.shape)
print("level_2_code kosong di CSV:", int(df["level_2_code"].isna().sum()))
print("Contoh pembentukan kode:")
display(df[[c for c in ["level_4_full_code", "level_6_full_code", "level_2_code"] if c in df.columns]].head(10))

for k in KEYS:
    assert k in df.columns, f"Kolom key '{k}' tidak ada di CSV"

# Pastikan kolom yang akan ditulis memang tersedia di DB.
skip_cols = [c for c in df.columns if c not in db_cols]
if skip_cols:
    print("PERINGATAN - kolom CSV tidak ada di DB (dilewati):", skip_cols)

special = {STATUS_COL, KET_COL, FLAG_COL}
data_cols = [c for c in df.columns if c in db_cols and c not in pk_cols and c not in special]
if "level_2_code" not in data_cols:
    raise RuntimeError(
        "level_2_code tidak masuk data_cols. "
        f"Kolom DB tersedia: {[c for c in db_cols if 'level' in c.lower()]}; "
        f"Kolom CSV tersedia: {[c for c in df.columns if 'level' in c.lower()]}"
    )

print("level_2_code ikut diproses:", "level_2_code" in data_cols)
print("Kolom yang akan ditulis:", len(data_cols))
df.head()


Ukuran CSV: (344659, 25)
level_2_code kosong di CSV: 0
Contoh pembentukan kode:


,level_4_full_code,level_2_code
0,1602022006,1602
1,1608010001,1608
2,1671020013,1671
3,1603033006,1603
4,1673020007,1673
5,1609010011,1609
6,1611030008,1611
7,1671062003,1671
8,1605090006,1605
9,1610052005,1610


PERINGATAN - kolom CSV tidak ada di DB (dilewati): ['level_4_full_code', 'is_remark_diperbaiki', 'is_remark_sesuai_lapangan', 'link_fasih_sm']
level_2_code ikut diproses: True
Kolom yang akan ditulis: 21


,assignment_id,level_4_full_code,nama_usaha,kategori,kode_kbli,metrik_tenaga_kerja,metrik_upah_gaji,metrik_biaya_produksi,metrik_biaya_pembelian,metrik_biaya_operasional,metrik_biaya_non_operasional,metrik_total_pengeluaran,metrik_pendapatan_barang_jasa,metrik_pendapatan_lainnya,metrik_nilai_tambah,metrik_total_aset,metrik_output,rasio_biaya_pembelian_barang_terhadap_omzet,rasio_ntb,produktivitas,flag_rasio_3_ntb_output,is_remark_diperbaiki,is_remark_sesuai_lapangan,link_fasih_sm,level_2_code
0,000009a7-fe00-41e5-bee2-0cc07cc65dc4,1602022006,USAHA AIR REBUSAN <NURHIDAYAH>,E,36001,1.0,0.0,500000.0,0.0,0.0,0.0,500000.0,1100000.0,0.0,600000.0,600000.0,1100000.0,0.0,0.545455,1100000.0,1,0,0,https://fasih-sm.bps.go.id/app/assignment/fd68...,1602
1,00000ee5-68a2-4eb2-ac93-2234dc24489d,1608010001,WARUNG SEMBAKO (RONAL),G,47112,1.0,0.0,200000.0,10000000.0,200000.0,150000.0,10550000.0,14000000.0,0.0,3600000.0,7000000.0,4000000.0,0.714286,0.9,4000000.0,1,0,0,https://fasih-sm.bps.go.id/app/assignment/fd68...,1608
2,00004c39-e36d-4e08-8acc-4270d009ce67,1671020013,WARUNG HODIJAH,G,47112,1.0,0.0,420000.0,8400000.0,650000.0,0.0,9470000.0,27470000.0,0.0,18000000.0,3400000.0,19070000.0,0.305788,0.943891,19070000.0,1,0,0,https://fasih-sm.bps.go.id/app/assignment/fd68...,1671
3,00010432-0be4-40b5-be6a-ebc6c05070e8,1603033006,Warung manisan (jajang suparman),G,47112,2.0,0.0,1200000.0,360000000.0,4000000.0,0.0,365200000.0,414000000.0,1000000.0,48800000.0,60000000.0,54000000.0,0.869565,0.903704,27000000.0,1,0,0,https://fasih-sm.bps.go.id/app/assignment/fd68...,1603
4,00012661-ab08-436b-af8a-75537385a0a9,1673020007,Konter pulsa,G,47901,3.0,700000.0,30000000.0,0.0,200000.0,0.0,30900000.0,45000000.0,0.0,14800000.0,22000000.0,45000000.0,0.0,0.328889,15000000.0,1,0,0,https://fasih-sm.bps.go.id/app/assignment/fd68...,1673


In [38]:
# Index supaya pencarian per baris tidak full-scan
key_sql = ", ".join(f'"{k}"' for k in KEYS)
cur.execute(f'CREATE INDEX IF NOT EXISTS idx_keys_flag3 ON "{TABEL}" ({key_sql})')
conn.commit()

where_sql = " AND ".join(f'"{k}" IS ?' for k in KEYS)
punya_updated = UPDATED_COL in db_cols
updated_expr = f'"{UPDATED_COL}"' if punya_updated else "NULL"

n_update = n_insert = n_pertahankan = 0

try:
    for _, rec in df.iterrows():
        row = {c: (None if pd.isna(rec[c]) else str(rec[c])) for c in df.columns}
        # Pastikan kode yang disimpan tidak berupa string kosong/spasi
        if row.get("level_2_code") is not None:
            row["level_2_code"] = row["level_2_code"].strip() or None

        key_vals = [row[k] for k in KEYS]

        found = cur.execute(
            f'SELECT rowid, "{STATUS_COL}", {updated_expr} '
            f'FROM "{TABEL}" WHERE {where_sql}',
            key_vals,
        ).fetchall()

        if found:
            for rid, old_status, old_updated in found:
                nilai = {c: row.get(c) for c in data_cols}

                # status & keterangan TIDAK diubah (tetap seperti di DB).
                # Kalau sebelumnya selesai (1): pertahankan juga tanggal update lama.
                if str(old_status).strip() in ("1", "1.0"):
                    if punya_updated:
                        nilai[UPDATED_COL] = old_updated
                    n_pertahankan += 1

                set_cols = list(nilai) + [FLAG_COL]
                vals = list(nilai.values()) + [1]
                set_sql = ", ".join(f'"{c}" = ?' for c in set_cols)
                cur.execute(
                    f'UPDATE "{TABEL}" SET {set_sql} WHERE rowid = ?',
                    vals + [rid],
                )
                n_update += 1
        else:
            ins_cols = data_cols + [STATUS_COL, KET_COL, FLAG_COL]
            vals = [row.get(c) for c in data_cols] + [0, None, 1]
            cols_sql = ", ".join(f'"{c}"' for c in ins_cols)
            ph = ", ".join("?" for _ in ins_cols)
            cur.execute(
                f'INSERT INTO "{TABEL}" ({cols_sql}) VALUES ({ph})',
                vals,
            )
            n_insert += 1

    conn.commit()
    print(
        f"Selesai. Update: {n_update} | Insert baru: {n_insert} | "
        f"Tanggal update lama dipertahankan: {n_pertahankan}"
    )
except Exception:
    conn.rollback()
    print("Error - perubahan pada transaksi ini di-rollback")
    raise

Selesai. Update: 270640 | Insert baru: 74258 | Tanggal update lama dipertahankan: 1491


In [40]:
# Verifikasi
keys_q = ", ".join(f'"{k}"' for k in KEYS)

print("flag_3_new = 1 :", cur.execute(f'SELECT COUNT(*) FROM "{TABEL}" WHERE "{FLAG_COL}" = 1').fetchone()[0],
      "| baris CSV:", len(df), "| key unik CSV:", len(df.drop_duplicates(KEYS)))
print("Total baris DB :", cur.execute(f'SELECT COUNT(*) FROM "{TABEL}"').fetchone()[0])

dup_db = pd.read_sql(
    f'SELECT {keys_q}, COUNT(*) AS jml FROM "{TABEL}" WHERE "{FLAG_COL}" = 1 '
    f'GROUP BY {keys_q} HAVING COUNT(*) > 1', conn)
print("Key duplikat di DB (flag=1):", len(dup_db))

display(pd.read_sql(
    f'SELECT {keys_q}, "{STATUS_COL}", "{KET_COL}", "{FLAG_COL}" '
    f'FROM "{TABEL}" WHERE "{FLAG_COL}" = 1 AND "{KET_COL}" IS NOT NULL LIMIT 10', conn))
conn.close()

ProgrammingError: Cannot operate on a closed database.

In [42]:
# Head DB (koneksi sendiri, bisa dijalankan kapan saja)
pd.set_option("display.max_columns", None)
with sqlite3.connect(DB_PATH) as c:
    df_db = pd.read_sql(f'SELECT * FROM "{TABEL}" WHERE link IS NULL', c)
display(df_db)

,link,assignment_id,level_2_code,kategori,kode_kbli,nama_usaha,metrik_tenaga_kerja,metrik_upah_gaji,metrik_biaya_produksi,metrik_biaya_pembelian,metrik_biaya_operasional,metrik_biaya_non_operasional,metrik_total_pengeluaran,metrik_pendapatan_barang_jasa,metrik_pendapatan_lainnya,metrik_nilai_tambah,metrik_total_aset,metrik_output,rasio_biaya_pembelian_barang_terhadap_omzet,reklasifikasi_skala_usaha,rasio_ntb,produktivitas,flag_rasio_1_output_aset,flag_rasio_2_upah_ntb,flag_rasio_3_ntb_output,flag_kategori_P_dan_U,sumber_file,status_penyelesaian,keterangan,update_at,flag_3_new
0,None,00010432-0be4-40b5-be6a-ebc6c05070e8,1603,G,47112,Warung manisan (jajang suparman),2,0,1200000.0,360000000,4000000.0,0,365200000.0,414000000,1000000,48800000.0,60000000.0,54000000.0,0.869565,None,0.903704,27000000.0,None,None,1,None,None,0,NaN,NaN,1
1,None,00030306-32f2-41c5-b9b1-d090adfc114b,1609,G,47723,JUAL OBAT KELILING (EXWAN),1,0,100000.0,24000000,4800000.0,0,28900000.0,48000000,0,19100000.0,6500000.0,24000000.0,0.500000,None,0.795833,24000000.0,None,None,1,None,None,0,NaN,NaN,1
2,None,0003e7ce-7287-4a4f-826d-fce421428d6e,1606,G,47111,WARUNG PUTRI BUNGSU,1,0,0.0,3000000,100000.0,0,3100000.0,4000000,0,900000.0,110000000.0,1000000.0,0.750000,None,0.900000,1000000.0,None,None,1,None,None,0,NaN,NaN,1
3,None,0004dbb8-ab11-45e9-9cb5-64d12d62c440,1604,I,56101,WARUNG GORENGAN DAN ES (ANI RATNAINI ),1,0,400000.0,0,0.0,0,400000.0,66000000,0,65600000.0,6000000.0,66000000.0,0.000000,None,0.993939,66000000.0,None,None,1,None,None,0,NaN,NaN,1
4,None,0004dbb8-ab11-45e9-9cb5-64d12d62c440,1604,C,33141,BENGKEL DINAMO (WAHYUDI),1,0,3500000.0,0,1800000.0,0,5300000.0,30000000,0,24700000.0,12000000.0,30000000.0,0.000000,None,0.823333,30000000.0,None,None,1,None,None,0,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74253,None,fffbe95d-3ef0-4eee-ab31-1e28e0a1e919,1610,G,47212,PEDAGANG BUAH HIDAYAT,2,3600000,0.0,20500000,4000000.0,2000000,30100000.0,38500000,2000000,14000000.0,0.0,18000000.0,0.532468,None,0.777778,9000000.0,None,None,1,None,None,0,NaN,NaN,1
74254,None,fffc56d6-dd10-44b8-9691-a71a16d5b524,1602,G,47241,PEDAGANG BERAS YAMDI PASAR KAMPUNG BARU,1,0,0.0,32000000,6000000.0,0,38000000.0,40000000,0,2000000.0,0.0,8000000.0,0.800000,None,0.250000,8000000.0,None,None,1,None,None,0,NaN,NaN,1
74255,None,fffc6935-01cf-4dcb-af2d-ca6ccadf4b62,1602,G,47112,TOKO SEMBAKO,1,0,0.0,24000000,3600000.0,1200000,28800000.0,139200000,0,111600000.0,50000000.0,115200000.0,0.172414,None,0.968750,115200000.0,None,None,1,None,None,0,NaN,NaN,1
74256,None,fffe896e-354a-43e5-b7ce-888dfdd9e631,1671,G,47591,TOKO JEVAN FURNITURE,3,24000000,0.0,30000000,5000000.0,0,59000000.0,60000000,0,25000000.0,0.0,30000000.0,0.500000,None,0.833333,10000000.0,None,None,1,None,None,0,NaN,NaN,1


In [21]:
with sqlite3.connect(DB_PATH) as c:
    c.execute("BEGIN")

    query = f"""
        DELETE FROM "{TABEL}"
        WHERE rowid IN (
            SELECT rowid
            FROM (
                SELECT
                    rowid,
                    ROW_NUMBER() OVER (
                        PARTITION BY "{KEYS[0]}", "{KEYS[1]}"
                        ORDER BY rowid
                    ) AS rn
                FROM "{TABEL}"
                WHERE "{FLAG_COL}" = 1
            )
            WHERE rn > 1
        )
    """

    c.execute(query)
    jumlah_hapus = c.execute("SELECT changes()").fetchone()[0]
    c.commit()

print(f"Jumlah baris duplikat yang dihapus: {jumlah_hapus}")

Jumlah baris duplikat yang dihapus: 171


In [43]:
LINK_COL = "link"
BASE_LINK = "https://fasih-sm.bps.go.id/app/assignment/fd68e454-ba45-4b85-8205-f3bf777ded24/"

kosong = f'("{LINK_COL}" IS NULL OR TRIM("{LINK_COL}") = \'\')'
ada_id = '"assignment_id" IS NOT NULL AND TRIM("assignment_id") <> \'\''

with sqlite3.connect(DB_PATH) as c:
    sebelum = c.execute(f'SELECT COUNT(*) FROM "{TABEL}" WHERE {kosong}').fetchone()[0]
    print("Link kosong sebelum:", sebelum)

    cur = c.execute(
        f'UPDATE "{TABEL}" SET "{LINK_COL}" = ? || TRIM("assignment_id") '
        f'WHERE {kosong} AND {ada_id}',
        (BASE_LINK,),
    )
    print("Link diisi:", cur.rowcount)

    sesudah = c.execute(f'SELECT COUNT(*) FROM "{TABEL}" WHERE {kosong}').fetchone()[0]
    print("Link kosong sesudah:", sesudah, "(tersisa karena assignment_id kosong)")

    display(pd.read_sql(
        f'SELECT "assignment_id", "{LINK_COL}" FROM "{TABEL}" '
        f'WHERE "{FLAG_COL}" = 1 LIMIT 5', c))

Link kosong sebelum: 74258
Link diisi: 74258
Link kosong sesudah: 0 (tersisa karena assignment_id kosong)


,assignment_id,link
0,30102a07-2cd5-44c4-8163-556891992c8d,https://fasih-sm.bps.go.id/app/assignment/fd68...
1,c927197f-c744-49dc-b5a7-597627bb3fb7,https://fasih-sm.bps.go.id/app/assignment/fd68...
2,fbaed23c-cf47-4904-9122-da93084c67d1,https://fasih-sm.bps.go.id/app/assignment/fd68...
3,1f5170c8-81ad-4dfb-8946-dbc95ebf9fac,https://fasih-sm.bps.go.id/app/assignment/fd68...
4,85e93c9a-40ed-4331-9a99-5e8f10bfb9a7,https://fasih-sm.bps.go.id/app/assignment/fd68...
